# Voting in Authoritarian Elections

**Authors:** Turkuler Isiksel, Thomas B. Pepinsky

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup & Data Loading

Load datasets from BigQuery using bigrquery. The dataset is public,
but you need a GCP project for billing.

Set `project_id` to your own GCP project ID below.

In [ ]:
# Install required packages (uncomment if needed)
# install.packages(c("bigrquery", "DBI"))

library(bigrquery)
library(DBI)

# Set your GCP project ID for billing
project_id <- "polirep-app"  # Change this to your GCP project

In [ ]:
# Load GWF Autocratic Regimes.csv
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwf_autocratic_regimes`"
gwf_autocratic_regimes <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("gwf_autocratic_regimes: %d rows, %d columns\n", nrow(gwf_autocratic_regimes), ncol(gwf_autocratic_regimes)))
head(gwf_autocratic_regimes)

In [ ]:
# Load GWF_AllPoliticalRegimes.parquet
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwf_allpoliticalregimes`"
gwf_allpoliticalregimes <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("gwf_allpoliticalregimes: %d rows, %d columns\n", nrow(gwf_allpoliticalregimes), ncol(gwf_allpoliticalregimes)))
head(gwf_allpoliticalregimes)

In [ ]:
# Load GWFcases.parquet
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwfcases`"
gwfcases <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("gwfcases: %d rows, %d columns\n", nrow(gwfcases), ncol(gwfcases)))
head(gwfcases)

In [ ]:
# Load GWFtscs.parquet
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_gwftscs`"
gwftscs <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("gwftscs: %d rows, %d columns\n", nrow(gwftscs), ncol(gwftscs)))
head(gwftscs)

In [ ]:
# Load institutions in dictatorships, 1946-2008.csv
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_institutions_in_dictatorships__1946_2008`"
institutions_in_dictatorships__1946_2008 <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("institutions_in_dictatorships__1946_2008: %d rows, %d columns\n", nrow(institutions_in_dictatorships__1946_2008), ncol(institutions_in_dictatorships__1946_2008)))
head(institutions_in_dictatorships__1946_2008)

In [ ]:
# Load institutions in dictatorships, 1946-2009.parquet
sql <- "SELECT * FROM `polirep-app.paper_data.replication_data_for_voting_in_authoritarian_elections_institutions_in_dictatorships__1946_2009`"
institutions_in_dictatorships__1946_2009 <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("institutions_in_dictatorships__1946_2009: %d rows, %d columns\n", nrow(institutions_in_dictatorships__1946_2009), ncol(institutions_in_dictatorships__1946_2009)))
head(institutions_in_dictatorships__1946_2009)

## Data Preparation


### Load GWF Cases Data

Load the regime case-level data from GWFcases. This dataset contains one
row per autocratic regime spell with start/end dates and regime characteristics.

In [ ]:
source("r_clean/scripts/00_setup.R")
gwf_cases <- load_data("GWFcases")
cat("GWFcases dimensions:", dim(gwf_cases), "\n")
cat("Regime types:\n")
print(table(gwf_cases$gwf_regimetype))
cat("Year range:", range(gwf_cases$gwf_startyr, na.rm=TRUE), "-", range(gwf_cases$gwf_endyr, na.rm=TRUE), "\n")
cat("Missing end year (censored):", sum(is.na(gwf_cases$gwf_endyr)), "\n")
head(gwf_cases)


### Expand Cases to TSCS Format

Transform GWFcases from case-level to time-series cross-sectional format
by expanding each regime case into one row per year in power. Creates
sequential year variable, duration, and failure indicators.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("GWFtscs dimensions:", nrow(gwf_tscs_final), "x", ncol(gwf_tscs_final), "\n")
cat("Countries:", length(unique(gwf_tscs_final$cowcode)), "\n")
cat("Year range:", range(gwf_tscs_final$year), "\n")
cat("Total failures:", sum(gwf_tscs_final$gwf_fail), "\n")
cat("Duration range:", range(gwf_tscs_final$duration), "years\n")
head(gwf_tscs_final[, c('cowcode', 'year', 'gwf_regimetype', 'duration', 'gwf_fail')], 10)


### Create Regime Type Dummies

Create collapsed binary indicators for party, personal, military, and
monarchy regimes. Includes special handling for Iran post-1980 (coded
as party regime despite being a theocracy).

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("Party regimes:", sum(gwf_tscs_final$gwf_party), "obs (", mean(gwf_tscs_final$gwf_party), ")\n")
cat("Personal regimes:", sum(gwf_tscs_final$gwf_personal, na.rm=TRUE), "obs (", mean(gwf_tscs_final$gwf_personal, na.rm=TRUE), ")\n")
cat("Military regimes:", sum(gwf_tscs_final$gwf_military, na.rm=TRUE), "obs (", mean(gwf_tscs_final$gwf_military, na.rm=TRUE), ")\n")
cat("Monarchies:", sum(gwf_tscs_final$gwf_monarch, na.rm=TRUE), "obs (", mean(gwf_tscs_final$gwf_monarch, na.rm=TRUE), ")\n")
head(gwf_tscs_final[, c('gwf_regimetype', 'gwf_party', 'gwf_personal', 'gwf_military', 'gwf_monarch')], 10)


### Calculate Past Regime Failures

Compute cumulative count of prior regime failures within each country.
Uses lag(cumsum(gwf_fail)) grouped by country code to track regime
instability history.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("Max past failures:", max(gwf_tscs_final$gwf_pastregimefail), "\n")
cat("Mean past failures:", mean(gwf_tscs_final$gwf_pastregimefail), "\n")
countries_5plus <- sum(aggregate(gwf_pastregimefail ~ cowcode, data=gwf_tscs_final, max)$gwf_pastregimefail >= 5)
cat("Countries with 5+ past failures:", countries_5plus, "\n")
head(gwf_tscs_final[, c('cowcode', 'year', 'gwf_fail', 'gwf_pastregimefail')], 20)


### Load GWF All Political Regimes

Load the comprehensive TSCS dataset containing all political regimes
(both autocracies and democracies) from 1946-2010. This is the base
dataset for Figure 1 analysis.

In [ ]:
source("r_clean/scripts/00_setup.R")
gwf_all <- load_data("GWF_AllPoliticalRegimes")
cat("GWF All dimensions:", dim(gwf_all), "\n")
cat("Countries:", length(unique(gwf_all$cowcode)), "\n")
cat("Year range:", range(gwf_all$year), "\n")
cat("Regime types:\n")
print(table(gwf_all$gwf_regimetype, useNA="ifany"))
cat("Democracies:", sum(gwf_all$gwf_nonautocracy == "democracy", na.rm=TRUE), "\n")
head(gwf_all)


### Load Svolik Institutions Data

Load the Svolik (2012) authoritarian institutions dataset containing
information on executive selection, legislative characteristics, party
systems, and military characteristics for dictatorships.

In [ ]:
source("r_clean/scripts/00_setup.R")
svolik <- load_data("institutions in dictatorships, 1946-2008")
cat("Svolik dimensions:", dim(svolik), "\n")
cat("Countries:", length(unique(svolik$ccode)), "\n")
cat("Year range:", range(svolik$year), "\n")
cat("Executive categories:\n")
print(table(svolik$executive))
cat("Legislative categories:\n")
print(table(svolik$legislative))
head(svolik)


### Merge GWF and Svolik Data

Merge GWF All Political Regimes with Svolik institutions data using
1:1 merge on country code (ccode) and year. This combines regime type
classifications with institutional characteristics.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_figure1.R")
cat("Merged dimensions:", nrow(merged_data), "\n")
cat("Year range:", range(merged_data$year), "\n")
head(merged_data[, c('ccode', 'year', 'gwf_regimetype', 'gwf_nonautocracy', 'executive', 'legislative')], 10)


### Create Regime Categories

Create three mutually exclusive regime categories: democracy (GWF coded),
electoral authoritarian (dictatorship with elected executive OR legislature
with parties), and closed authoritarian (dictatorship without elections).

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_figure1.R")
cat("Mean democracy:", mean(regime_data$democracy, na.rm=TRUE), "\n")
cat("Mean electoral auth:", mean(regime_data$el_auth, na.rm=TRUE), "\n")
cat("Mean closed auth:", mean(regime_data$nonel_auth, na.rm=TRUE), "\n")
head(regime_data[, c('ccode', 'year', 'democracy', 'dictatorship', 'el_auth', 'nonel_auth')], 20)


### Aggregate to Year-Level Proportions

Collapse the country-year data to year-level means, calculating the
proportion of all consolidated regimes that are democratic, electoral
authoritarian, or closed authoritarian in each year (1946-2008).

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/02_figure1.R")
cat("Years:", nrow(regime_proportions), "\n")
cat("Year range:", range(regime_proportions$year), "\n")
cat("\nFirst 10 years:\n")
print(head(regime_proportions, 10))
cat("\nLast 10 years:\n")
print(tail(regime_proportions, 10))


## Data Cleaning


### Data Cleaning

Transform GWFcases to time-series cross-sectional (TSCS) format with regime failures and regime type dummies (implements clean.do)

## Figure 1: Electoralism, 1946-2008


### Figure 1: Electoralism, 1946-2008

Time trends in regime types: democratic, electoral authoritarian, and closed authoritarian regimes (implements figure1.do)